# SQD on the (22e,16o) [2Fe–2S] active space — 32 qubitsSelf-contained. Nothing to clone; you upload one 308 KB file.**What this computes:** SQD energies at several subspace sizes, against the exact CASCIreference `-5013.72724223 Ha` (already computed classically — 19,079,424 determinants, 2.57 h).**Runtime warning.** The cost is the classical selected-CI solve, which is **CPU-bound**.A GPU/H100 runtime does *not* help — a hosted notebook gives ~8–12 vCPUs either way,
so pick a**standard CPU runtime** and save the GPU quota. Measured on 8 cores: `spb=1000` took**4.4 h** for a single point. The `spb` values below are chosen to fit a session:| samples/batch | subspace (approx) | expected ||---|---|---|| 200 | ~160,000 | ~5 min || 400 | ~640,000 | ~30 min || 800 | ~2,600,000 | ~2.5 h |Results are written to disk **after every point**, so a disconnect costs you at most one point.

In [ ]:
!pip -q install "pyscf>=2.9" "qiskit>=1.3,<3" "qiskit-addon-sqd>=0.13" "ffsim>=0.0.83"import sys; print(sys.version)

## 1. The ffsim patchAn upstream bug in ffsim (0.0.83 / 0.0.84) breaks `optimize=True`, which the IBMtutorial's LUCJ ansatz relies on: `unitaries_to_parameters` calls `scipy.linalg.logm`on a batch of matrices, but `logm` only accepts one square matrix. Without the fix theansatz stays concentrated on a single configuration and SQD lands hundreds of mHa off.This installs the fix only if the bug is actually present.

In [ ]:
import numpy as np, scipy.linalgimport ffsim.linalg.util as _udef _batched(mats, real=False):    return _u.antihermitians_to_parameters(        np.stack([scipy.linalg.logm(m) for m in np.asarray(mats)]), real=real)try:    _u.unitaries_to_parameters(np.eye(2)[None, :]); print("upstream already fixed")except ValueError:    for name in ("ffsim.linalg.util","ffsim.variational.ucj_spin_balanced",                 "ffsim.variational.ucj_spin_unbalanced","ffsim.variational.util"):        try: m = __import__(name, fromlist=["_"])        except ImportError: continue        if hasattr(m,"unitaries_to_parameters"): m.unitaries_to_parameters = _batched        if hasattr(m,"unitary_to_parameters"):            m.unitary_to_parameters = lambda x, real=False: _batched(np.asarray(x)[None,:], real)    _u.unitaries_to_parameters(np.eye(2)[None, :])    print("patch installed and verified")

## 2. Upload the HamiltonianUpload **`results/stage1_sto-3g_fe3d+brs3p.npz`** (308 KB) from the project.It holds the active-space integrals and the CCSD amplitudes that seed the ansatz.

In [ ]:
# Upload results/stage1_sto-3g_fe3d+brs3p.npz (308 KB).
# On a hosted notebook the picker handles it; locally, just set NPZ to the path.
try:
    from google.colab import files  # hosted notebook
    up = files.upload()
    NPZ = next(k for k in up if k.endswith(".npz"))
except ImportError:
    NPZ = "stage1_sto-3g_fe3d+brs3p.npz"   # local / other host: edit this path
print("using", NPZ)

## 3. Build the LUCJ circuit (32 qubits)

In [ ]:
import ffsim, numpy as npfrom qiskit import QuantumCircuit, QuantumRegisterd = np.load(NPZ)hcore, eri = d["hcore"], d["eri"]e_nuc = float(d["nuclear_repulsion_energy"])norb  = hcore.shape[0]nelec = (11, 11)                      # (22e,16o)E_EXACT = -5013.72724223              # exact CASCI singlet, computed classicallyCAS_DIM = 19_079_424pairs_aa = [(p, p+1) for p in range(norb-1)]ucj = ffsim.UCJOpSpinBalanced.from_t_amplitudes(    t2=d["t2"], t1=d["t1"], n_reps=4,    interaction_pairs=(pairs_aa, None), optimize=True)qr = QuantumRegister(2*norb, "q")circuit = QuantumCircuit(qr)circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qr)circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj), qr)circuit.measure_all()tr = circuit.decompose(reps=4)print(f"{2*norb} qubits | depth {tr.depth()} | 2q gates {tr.count_ops().get('cx',0)}")print(f"CAS dimension {CAS_DIM:,} | exact reference {E_EXACT:.8f} Ha")

## 4. Sample the circuit (fast — this is not the bottleneck)

In [ ]:
SHOTS = 200_000sampler = ffsim.qiskit.FfsimSampler(default_shots=SHOTS, norb=norb, nelec=nelec,                                    global_depolarizing=0.0, seed=2026)bit_array = sampler.run([circuit]).result()[0].data.measbits = np.unpackbits(bit_array.array, axis=1, bitorder="big")[:, -2*norb:]na, nb = bits[:, norb:].sum(1), bits[:, :norb].sum(1)valid = ((na == nelec[0]) & (nb == nelec[1])).mean()print(f"{SHOTS:,} shots | {valid*100:.2f}% with correct N and Sz | "      f"{len({r.tobytes() for r in bit_array.array}):,} unique")

## 5. Run SQD — saves after every pointEach point writes `sqd_22e16o_results.json`. If the session disconnects you lose at mostthe point in flight. Start with `SPB_LIST = [200]` if you want a number in five minutes.

In [ ]:
import json, timefrom functools import partialfrom qiskit_addon_sqd.fermion import diagonalize_fermionic_hamiltonian, solve_sci_batchSPB_LIST = [200, 400, 800]        # trim this if the session is shortOUT = "sqd_22e16o_results.json"rows = []# max_cycle=500, not the tutorial's 200: at 200 the Davidson stops short by ~0.1 mHa# even on the full space, which would masquerade as a sampling error.sci_solver = partial(solve_sci_batch, spin_sq=0.0, max_cycle=500)for spb in SPB_LIST:    t0 = time.time()    res = diagonalize_fermionic_hamiltonian(        hcore, eri, bit_array, samples_per_batch=spb, norb=norb, nelec=nelec,        num_batches=3, energy_tol=1e-6, occupancies_tol=1e-5, max_iterations=5,        sci_solver=sci_solver, symmetrize_spin=True, carryover_threshold=1e-4,        seed=2026)    e = float(res.energy + e_nuc)    dim = int(np.prod(res.sci_state.amplitudes.shape))    row = {"samples_per_batch": spb, "energy": e,           "error_mha": (e - E_EXACT)*1e3, "subspace_dim": dim,           "pct_of_cas": 100*dim/CAS_DIM, "wall_s": time.time()-t0}    rows.append(row)    json.dump({"system":"(22e,16o) 32 qubits","e_exact":E_EXACT,               "cas_dim":CAS_DIM,"shots":SHOTS,"rows":rows}, open(OUT,"w"), indent=2)    print(f"spb={spb:<5d} dim={dim:>9,} ({row['pct_of_cas']:5.1f}% of CAS)  "          f"E={e:.8f}  err={row['error_mha']:+9.4f} mHa  [{row['wall_s']/60:.1f} min]")

## 6. Results

In [ ]:
import jsonr = json.load(open(OUT))print(f"exact CASCI        {r['e_exact']:.8f} Ha   ({r['cas_dim']:,} determinants)")print()print(f"{'spb':>6} {'subspace':>11} {'% of CAS':>9} {'error (mHa)':>13} {'min':>7}")for x in r["rows"]:    print(f"{x['samples_per_batch']:>6} {x['subspace_dim']:>11,} "          f"{x['pct_of_cas']:>8.1f}% {x['error_mha']:>13.4f} {x['wall_s']/60:>7.1f}")print()print("reference points already computed:")print("  (10e,10o) 20 qubits, 63,504 dets : SQD error +0.0000 mHa (exact)")print("  (22e,16o) 32 qubits, spb=1000    : SQD error +34.4783 mHa, subspace 5,716,881")

In [ ]:
try:
    from google.colab import files  # hosted notebook
    files.download(OUT)
except ImportError:
    print(f"results written to {OUT}")